# 🦗 03 — UMAP + HDBSCAN Clustering
โหลด feature matrix → ลด dimension ด้วย UMAP → Cluster ด้วย HDBSCAN

**เป้าหมาย:** ค้นพบโครงสร้างตามธรรมชาติของเสียงจิ้งหรีดโดยไม่ต้องกำหนด Label ล่วงหน้า


In [ ]:
import sys
sys.path.insert(0, "../src")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from cricket_perception.clustering import ClusterPipeline, summarise_clusters

plt.style.use("dark_background")
CACHE_DIR = Path("../features_cache")
RESULTS_DIR = Path("../results")
RESULTS_DIR.mkdir(exist_ok=True)
print("✅ Imports OK")


## 1. โหลด Feature Matrix

In [ ]:
X = np.load(CACHE_DIR / "features_X.npy")
df_meta = pd.read_csv(CACHE_DIR / "features_meta.csv")
print(f"Feature matrix: {X.shape}")
print(f"Segments: {len(df_meta)} | Species: {df_meta.species.nunique()}")

# Standardize (important for UMAP)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print("
✅ Standardized (mean≈0, std≈1)")


## 2. UMAP + HDBSCAN

In [ ]:
# ── Hyperparameters ──
# Adjust these to explore different granularities:
#   umap_n_neighbors: low (5-10) = fine-grained local, high (30-50) = global
#   hdbscan_min_cluster_size: low = many small clusters, high = fewer big ones

pipe = ClusterPipeline(
    umap_n_neighbors=15,
    umap_min_dist=0.1,
    umap_random_state=42,
    hdbscan_min_cluster_size=10,
    hdbscan_min_samples=5,
    backend="auto",   # will use cuML if GPU available
)

labels, embedding = pipe.fit(X_scaled)
df_meta["cluster"] = labels

n_clusters = (labels >= 0).sum()
n_noise    = (labels == -1).sum()
print(f"
✅ Clustering done!")
print(f"   Unique clusters: {len(set(labels[labels>=0]))}")
print(f"   Noise points:    {n_noise} ({n_noise/len(labels):.1%})")


## 3. Visualize Clusters

In [ ]:
# ── Color by Cluster ──
fig = pipe.plot(
    title="Cricket Soundscape Clusters (InsectSet32)",
    save_path=RESULTS_DIR / "03_clusters_hdbscan.png",
)
plt.show()


In [ ]:
# ── Color by Species (ground truth validation) ──
import seaborn as sns

species_list = df_meta.species.unique()
palette = sns.color_palette("husl", len(species_list))
sp_colors = {sp: palette[i] for i, sp in enumerate(species_list)}

fig, ax = plt.subplots(figsize=(11, 9))
fig.patch.set_facecolor("#0f0f1a")
ax.set_facecolor("#0f0f1a")
ax.tick_params(colors="white")

for sp in species_list:
    mask = df_meta.species == sp
    ax.scatter(embedding[mask, 0], embedding[mask, 1],
               c=[sp_colors[sp]], label=sp, alpha=0.6, s=15, edgecolors="none")

ax.legend(loc="upper right", framealpha=0.15, labelcolor="white",
          fontsize=7, ncol=2)
ax.set_title("UMAP — Colored by Species", color="white", fontsize=14)
ax.set_xlabel("UMAP-1", color="white")
ax.set_ylabel("UMAP-2", color="white")
plt.tight_layout()
plt.savefig(RESULTS_DIR / "03_clusters_by_species.png", dpi=150,
            bbox_inches="tight", facecolor=fig.get_facecolor())
plt.show()


## 4. Cluster Summary

In [ ]:
summary = summarise_clusters(
    labels,
    metadata=df_meta.to_dict("records"),
)

print(f"{"Cluster":>8} | {"Count":>6} | {"Frac":>6} | Sample Species")
print("-" * 60)
for lbl, info in sorted(summary.items()):
    tag = "(noise)" if info["is_noise"] else ""
    # Show which species are in this cluster
    mask = df_meta.cluster == lbl
    top_species = df_meta[mask].species.value_counts().index[:3].tolist()
    print(f"{lbl:>8} | {info["count"]:>6} | {info["fraction"]:>5.1%} | {top_species} {tag}")

print("
💡 Next: listen to samples from each cluster, label them, then run behavior analysis")


## 5. Save Pipeline & Results

In [ ]:
pipe.save(RESULTS_DIR / "cluster_pipeline.pkl")
df_meta.to_csv(RESULTS_DIR / "segments_with_clusters.csv", index=False)
print("✅ Saved pipeline and cluster labels")
print("
Files saved:")
for p in RESULTS_DIR.iterdir():
    print(f"  {p.name}")
